# Sparkfun Crawled EAGLE / KiCad Zip link

In [ ]:
import csv
import time
import requests
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode, urljoin
from lxml import html
from requests.adapters import HTTPAdapter, Retry

# ---------- HTTP session with retries ----------
def _session_with_retries(total=3, backoff=0.5, timeout=20):
    s = requests.Session()
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['HEAD','GET','OPTIONS'])
    )
    s.mount('http://', HTTPAdapter(max_retries=retries))
    s.mount('https://', HTTPAdapter(max_retries=retries))
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/124.0.0.0 Safari/537.36")
    })
    s.request_timeout = timeout
    return s

# ---------- Utilities ----------
def _normalize_to_page1(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs.pop('p', None)
    return urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

def _is_empty_page(tree) -> bool:
    """
    Detect Magento 'no products' message:
    <div class="message info empty"><div>We can't find products matching the selection.</div></div>
    """
    return bool(tree.xpath("//div[contains(@class,'message') and contains(@class,'info') and contains(@class,'empty')]"))

def _iter_pages_until_empty(category_url: str, sess: requests.Session, max_pages: int = 200):
    """
    Yield page URLs from ?p=1 increasing until an empty page is encountered,
    or until max_pages is reached (safety cap).
    """
    base_url = _normalize_to_page1(category_url)
    parsed = urlparse(base_url)

    for i in range(1, max_pages + 1):
        qs = parse_qs(parsed.query)
        qs['p'] = [str(i)]
        page_url = urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

        r = sess.get(page_url, timeout=sess.request_timeout)
        r.raise_for_status()
        tree = html.fromstring(r.content)

        if _is_empty_page(tree):
            break  # stop when Magento shows the empty message

        yield page_url

def _get_product_links_from_page(page_url: str, sess: requests.Session) -> list[str]:
    r = sess.get(page_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    anchors = tree.xpath('//*[@id="amasty-shopby-product-list"]/div[2]/ol//a[@href]')
    seen, out = set(), []
    for a in anchors:
        href = a.get('href')
        if href:
            absolute = urljoin(page_url, href)
            if absolute not in seen:
                seen.add(absolute)
                out.append(absolute)
    return out

def _find_kicad_eagle_links(product_url: str, sess: requests.Session):
    """
    Return (kicad_url, eagle_url)
    - kicad_url: .zip whose anchor text contains 'kicad'
    - eagle_url: .zip whose anchor text contains 'eagle', else 3rd .zip found if exists
    """
    r = sess.get(product_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    zip_links_in_order = []
    kicad_url = ""
    eagle_url = ""

    for a in tree.xpath('//a[@href]'):
        href = (a.get('href') or '').strip()
        if not href.lower().endswith('.zip'):
            continue
        abs_url = urljoin(product_url, href)
        zip_links_in_order.append(abs_url)

        text = (a.text_content() or '').strip().lower()
        if not kicad_url and "kicad" in text:
            kicad_url = abs_url
        if not eagle_url and "eagle" in text:
            eagle_url = abs_url

    # Deduplicate while preserving order
    seen = set()
    zip_links_in_order = [u for u in zip_links_in_order if not (u in seen or seen.add(u))]

    # Eagle fallback
    if not eagle_url and len(zip_links_in_order) >= 3:
        eagle_url = zip_links_in_order[2]

    return kicad_url, eagle_url

# ---------- Main Function ----------
def category_to_csv(category_url: str, csv_path: str, delay_sec: float = 0.0):
    """
    Crawl category URL and save CSV with columns:
    page_url, product_url, kicad_url, eagle_url
    Iterates pages sequentially (?p=1,2,3,...) until the 'empty' message appears.
    """
    sess = _session_with_retries()

    page_count = 0
    product_count = 0

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["page_url", "product_url", "kicad_url", "eagle_url"])

        for page_url in _iter_pages_until_empty(category_url, sess):
            page_count += 1
            try:
                product_links = _get_product_links_from_page(page_url, sess)
                product_count += len(product_links)
                print(f"[Page {page_count}] {page_url} -> {len(product_links)} products")
            except Exception as e:
                print(f"[Page error] {page_url} -> {e}")
                continue

            if delay_sec:
                time.sleep(delay_sec)

            for product_url in product_links:
                try:
                    kicad_url, eagle_url = _find_kicad_eagle_links(product_url, sess)
                except Exception as e:
                    print(f"[Product error] {product_url} -> {e}")
                    kicad_url, eagle_url = "", ""

                writer.writerow([page_url, product_url, kicad_url, eagle_url])

                if delay_sec:
                    time.sleep(delay_sec)

    print(f"Done. Pages crawled: {page_count}, products seen: {product_count}")


def csv_path_from_url(url, base_folder):
    """Convert category URL to CSV path inside base_folder."""
    path_name = os.path.basename(urlparse(url).path)  # e.g., "components.html"
    if path_name.endswith(".html"):
        path_name = path_name[:-5]  # remove ".html"
    file_name = f"sparkfun_{path_name}_zips.csv"
    return os.path.join(base_folder, file_name)

# Example

In [ ]:
sparkfun_category_url =['https://www.sparkfun.com/audio.html',
                        'https://www.sparkfun.com/components.html',
                        'https://www.sparkfun.com/data-logging-and-memory.html',
                        'https://www.sparkfun.com/development-boards.html',
                        'https://www.sparkfun.com/displays.html',
                        'https://www.sparkfun.com/e-textiles-crafting.html',
                        'https://www.sparkfun.com/gnss.html',
                        'https://www.sparkfun.com/iot-wireless.html',
                        'https://www.sparkfun.com/kits.html',
                        'https://www.sparkfun.com/robotics.html',
                        'https://www.sparkfun.com/sensors.html',
                        'https://www.sparkfun.com/tools.html',
                        'https://www.sparkfun.com/special-categories.html']

In [ ]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
cat_url = 'https://www.sparkfun.com/data-logging-and-memory.html'
category_to_csv(
    cat_url,
    csv_path_from_url(cat_url, base_folder),
    delay_sec=0.2        # polite throttling
)

In [ ]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"

for cat_url in sparkfun_category_url:
    category_to_csv(
        cat_url,
        csv_path_from_url(cat_url, base_folder),
        delay_sec=0.2        # polite throttling
    )

# Download Zip 

In [ ]:
import os
import csv
from collections import Counter
import pathlib
import requests
from urllib.parse import urlparse

def combine_csv_files(src_folder, output_csv):
    """
    Combine all CSV files in a folder into a single CSV file.
    Assumes all CSVs have the same header.

    Args:
        src_folder (str): Path to folder containing CSV files
        output_csv (str): Path to save the combined CSV
    """
    csv_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV files found in the source folder.")

    header_written = False

    with open(output_csv, "w", newline="", encoding="utf-8") as out_f:
        writer = None

        for csv_file in csv_files:
            csv_path = os.path.join(src_folder, csv_file)
            with open(csv_path, newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                header = next(reader)

                if not header_written:
                    writer = csv.writer(out_f)
                    writer.writerow(header)
                    header_written = True

                for row in reader:
                    writer.writerow(row)

    print(f"Combined {len(csv_files)} CSV files into: {output_csv}")




def get_zip_links_from_csv(csv_path, url_col_index=3):
    """
    Get a list of ZIP file links from a specific column of a CSV file.

    Args:
        csv_path (str): Path to the CSV file.
        url_col_index (int): Zero-based index of the column containing the URLs.
                             Default is 3 (fourth column).

    Returns:
        list[str]: List of ZIP file URLs from the specified column.
    """
    zip_links = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # skip header
        for row in reader:
            if len(row) > url_col_index:
                url = row[url_col_index].strip()
                if url.lower().endswith(".zip"):
                    zip_links.append(url)
    return zip_links




def find_duplicate_links(links):
    """
    Find duplicate links in a list.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List of duplicate links (each appears once in output)
    """
    counter = Counter(links)
    duplicates = [link for link, count in counter.items() if count > 1]
    return duplicates


def remove_duplicates(links):
    """
    Remove duplicate links from a list while preserving order.

    Args:
        links (list[str]): List of links

    Returns:
        list[str]: List with duplicates removed
    """
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    return unique_links


def download_zips_to_named_folders(links, dest_folder, overwrite=False, timeout=30):
    """
    Download ZIP files from a list of URLs and save each into its own subfolder.
    Folder name is based on the ZIP name (without extension) from the URL.
    The ZIP file inside is always saved as 'eagle.zip'.
    If folder exists and overwrite=False, add ##1, ##2... until unique.

    Args:
        links (list[str]): List of ZIP URLs
        dest_folder (str): Destination base folder
        overwrite (bool): Overwrite if file exists
        timeout (int): Timeout for download in seconds

    Returns:
        tuple: (list of saved paths, list of failed URLs)
    """
    os.makedirs(dest_folder, exist_ok=True)
    saved_paths = []
    failed_urls = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }

    total = len(links)
    for idx, url in enumerate(links, start=1):
        try:
            # Base folder name from ZIP file name (without .zip)
            base_folder_name = pathlib.Path(urlparse(url).path).stem or f"file_{idx}"

            # Ensure unique folder if overwrite=False
            zip_folder = os.path.join(dest_folder, base_folder_name)
            if not overwrite:
                counter = 1
                while os.path.exists(zip_folder):
                    zip_folder = os.path.join(dest_folder, f"{base_folder_name}##{counter}")
                    counter += 1

            os.makedirs(zip_folder, exist_ok=True)
            dest_path = os.path.join(zip_folder, "eagle.zip")

            print(f"[{idx}/{total}] Downloading: {url} -> {dest_path}")

            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp_path = dest_path + ".part"
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 64):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, dest_path)

            saved_paths.append(dest_path)

        except Exception as e:
            print(f"[{idx}/{total}] Failed to download {url}: {e}")
            failed_urls.append(url)

    # Summary report
    print("\n=== Download Summary ===")
    print(f"✅ Successful downloads: {len(saved_paths)}")
    print(f"❌ Failed downloads: {len(failed_urls)}")
    if failed_urls:
        print("Failed URLs:")
        for u in failed_urls:
            print("  -", u)

    return saved_paths, failed_urls



# Example

In [ ]:
src_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
output_csv = "/Users/taitinglu/Desktop/IMG2SCH/sparkfun_combined.csv"
dest_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Zip"

# combine_csv_files(src_folder, output_csv)

In [ ]:
links = get_zip_links_from_csv(output_csv)
duplicates = find_duplicate_links(links)
print(len(duplicates))
unique_links = remove_duplicates(links)
print("Unique links:", len(unique_links))
download_zips_to_named_folders(unique_links, dest_folder)

# Sparkfun Tutorial

In [ ]:
import csv
import requests
from lxml import html

UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

def _class_contains(x):
    return f"contains(concat(' ', normalize-space(@class), ' '), ' {x} ')"

def save_sparkfun_titles_and_urls(page_url, csv_path):
    """
    Scrape ONLY grid tiles from the given SparkFun tutorials page
    and save their title + URL into a CSV file.

    Args:
        page_url (str): URL of the SparkFun tutorials page (e.g., "?page=all")
        csv_path (str): Output CSV file path
    """
    r = requests.get(page_url, headers={"User-Agent": UA}, timeout=25)
    r.raise_for_status()

    tree = html.fromstring(r.content)

    container_xpath = f"//*[@id='airlock']//div[{_class_contains('tutorials-index')}]"
    containers = tree.xpath(container_xpath)
    if not containers:
        print("[WARN] tutorials-index container not found.")
        return
    container = containers[0]

    tiles_xpath = (
        f".//div[{_class_contains('tile')} and {_class_contains('tutorial-tile')} and {_class_contains('grid')}]"
    )
    tiles = container.xpath(tiles_xpath)

    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Title", "URL"])  # header

        for t in tiles:
            # Title
            title_nodes = t.xpath(".//h3[contains(@class,'title')]/text()")
            title = title_nodes[0].strip() if title_nodes else ""

            # URL
            a = t.xpath(".//a[@href]")
            url_t = a[0].get("href") if a else ""

            writer.writerow([title, url_t])

    print(f"[OK] Saved {len(tiles)} tutorials to {csv_path}")






In [ ]:
tutorial_url = "https://learn.sparkfun.com/tutorials?page=all"
tutorial_csv = '/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials.csv'
save_sparkfun_titles_and_urls(
    tutorial_url,
    tutorial_csv
)

In [ ]:
import csv
import re
import time
from typing import Tuple
from urllib.parse import urljoin

import requests
from lxml import html
from requests.adapters import HTTPAdapter, Retry

UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

def _session_with_retries(total=3, backoff=0.5, timeout=25) -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["HEAD", "GET", "OPTIONS"]),
        raise_on_status=False,
    )
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update({"User-Agent": UA, "Accept": "text/html,application/xhtml+xml"})
    s.request_timeout = timeout
    return s

def _fetch_html(session: requests.Session, url: str) -> str:
    resp = session.get(url, timeout=getattr(session, "request_timeout", 25))
    resp.raise_for_status()
    if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    return resp.text

def _extract_zip_links(html_text: str, base_url: str) -> Tuple[str, str]:
    tree = html.fromstring(html_text)
    anchors = tree.xpath("//a[@href]")
    zip_re = re.compile(r"\.zip(?:[?#].*)?$", re.IGNORECASE)

    eagle_url = ""
    kicad_url = ""

    for a in anchors:
        text = (a.text_content() or "").strip()
        href = a.get("href", "")
        if not href:
            continue
        abs_url = urljoin(base_url, href)
        if not zip_re.search(abs_url):
            continue

        low = text.lower()
        if not eagle_url and "eagle" in low:
            eagle_url = abs_url
        if not kicad_url and "kicad" in low:
            kicad_url = abs_url

        if eagle_url and kicad_url:
            break

    return eagle_url, kicad_url

def enrich_tutorial_csv_with_design_files(
    input_csv_path: str,
    output_csv_path: str,
    delay_sec: float = 0.2
):
    """
    Read CSV with headers Title,URL and write a new CSV with:
    Title,URL,eagle_zip_url,kicad_zip_url
    Prints an index while processing.
    """
    session = _session_with_retries()

    with open(input_csv_path, "r", encoding="utf-8-sig", newline="") as infile, \
         open(output_csv_path, "w", encoding="utf-8", newline="") as outfile:

        reader = list(csv.DictReader(infile))  # convert to list so we can get total length
        total = len(reader)

        fieldnames = ["Title", "URL", "eagle_zip_url", "kicad_zip_url"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for idx, row in enumerate(reader, start=1):
            title = row.get("Title", "").strip()
            url = row.get("URL", "").strip()
            eagle_zip = ""
            kicad_zip = ""

            print(f"[{idx}/{total}] Processing: {title or '(no title)'}")

            if url:
                try:
                    html_text = _fetch_html(session, url)
                    eagle_zip, kicad_zip = _extract_zip_links(html_text, url)
                except Exception as e:
                    print(f"  [WARN] Failed to fetch/extract for {url} -> {e}")
                time.sleep(delay_sec)

            writer.writerow({
                "Title": title,
                "URL": url,
                "eagle_zip_url": eagle_zip,
                "kicad_zip_url": kicad_zip
            })

    print(f"[OK] Wrote enriched CSV to: {output_csv_path}")





# Example

In [ ]:
# Example usage:
if __name__ == "__main__":
    enrich_tutorial_csv_with_design_files(
        input_csv_path="/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials.csv",          # must have Title,URL
        output_csv_path="/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials_with_zips.csv",
        delay_sec=0.25
    )

In [ ]:
output_csv = '/Users/taitinglu/Desktop/IMG2SCH/sparkfun_tutorials_with_zips.csv'
dest_folder = '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial Zip'

links = get_zip_links_from_csv(output_csv,url_col_index=2)
duplicates = find_duplicate_links(links)
print("duplicate: ",len(duplicates))
unique_links = remove_duplicates(links)
print("Unique links:", len(unique_links))
download_zips_to_named_folders(unique_links, dest_folder)

# Merge all sch file

In [9]:
import os
import xml.etree.ElementTree as ET
import shutil
import filecmp




def copy_all_files_from_folders(folder_list, dest_folder, recursive=False):
    """
    Copy all files from a list of folders into a single destination folder.

    Args:
        folder_list (list[str]): List of source folder paths
        dest_folder (str): Destination folder path
        recursive (bool): If True, include files from subfolders too
    """
    os.makedirs(dest_folder, exist_ok=True)
    count = 0

    for folder in folder_list:
        if not os.path.isdir(folder):
            print(f"Skipping (not a folder): {folder}")
            continue

        if recursive:
            for root, _, files in os.walk(folder):
                for file in files:
                    src_path = os.path.join(root, file)
                    dst_path = os.path.join(dest_folder, file)
                    # Avoid overwriting by adding ##N if needed
                    counter = 1
                    while os.path.exists(dst_path):
                        name, ext = os.path.splitext(file)
                        dst_path = os.path.join(dest_folder, f"{name}##{counter}{ext}")
                        counter += 1
                    shutil.copy2(src_path, dst_path)
                    count += 1
        else:
            for file in os.listdir(folder):
                src_path = os.path.join(folder, file)
                if os.path.isfile(src_path):
                    dst_path = os.path.join(dest_folder, file)
                    # Avoid overwriting by adding ##N if needed
                    counter = 1
                    while os.path.exists(dst_path):
                        name, ext = os.path.splitext(file)
                        dst_path = os.path.join(dest_folder, f"{name}##{counter}{ext}")
                        counter += 1
                    shutil.copy2(src_path, dst_path)
                    count += 1

    print(f"Copied {count} file(s) to {dest_folder}")



# Example

In [27]:
folders = [
    '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Github Sch UnZip',
    '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Category Sch UnZip',
    '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial Sch UnZip'
]
copy_all_files_from_folders(folders, '/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Sch UnZip', recursive=False)

Copied 1292 file(s) to /Users/taitinglu/Desktop/IMG2SCH/Sparkfun Sch UnZip


# Filter duplicate Sch

In [10]:
import os
import shutil
import hashlib
import xml.etree.ElementTree as ET




def _xml_semantic_fingerprint(path,
                              ignore_whitespace=True,
                              ignore_comments=True,
                              ignore_child_order=False):
    """
    Build a canonical tuple for the XML and return a stable hash digest.
    """
    def norm_text(s):
        if s is None:
            return ""
        return " ".join(s.split()) if ignore_whitespace else s

    def canonical(elem):
        tag = elem.tag
        attrs = tuple(sorted((k, norm_text(v)) for k, v in elem.attrib.items()))
        text = norm_text(elem.text)
        children = []
        for child in list(elem):
            if ignore_comments and child.tag is ET.Comment:
                continue
            children.append(canonical(child))
        if ignore_child_order:
            children.sort()
        return (tag, attrs, text, tuple(children))

    tree = ET.parse(path)
    root_tuple = canonical(tree.getroot())
    return hashlib.sha256(repr(root_tuple).encode("utf-8")).hexdigest()


def copy_unique_xml_files_with_semantic_compare(src_folder: str,
                                                dest_folder: str,
                                                ext=".sch",
                                                ignore_whitespace=True,
                                                ignore_comments=True,
                                                ignore_child_order=False):
    """
    Copy XML-like files (default: .sch) from src_folder to dest_folder,
    skipping if a semantically identical file already exists in dest_folder,
    regardless of filename. Uses an in-memory fingerprint cache for speed.
    """
    if not os.path.isdir(src_folder):
        raise ValueError(f"Source folder not found: {src_folder}")
    os.makedirs(dest_folder, exist_ok=True)

    # Collect source files
    src_files = [f for f in os.listdir(src_folder) if f.lower().endswith(ext.lower())]
    total = len(src_files)

    # Build destination fingerprint cache
    dest_files = [f for f in os.listdir(dest_folder) if f.lower().endswith(ext.lower())]
    dest_fingerprints = set()
    for f in dest_files:
        dest_path = os.path.join(dest_folder, f)
        try:
            fp = _xml_semantic_fingerprint(dest_path,
                                           ignore_whitespace=ignore_whitespace,
                                           ignore_comments=ignore_comments,
                                           ignore_child_order=ignore_child_order)
            dest_fingerprints.add(fp)
        except Exception as e:
            print(f"  Warning: could not hash dest file {dest_path}: {e}")

    copied_count = 0
    skipped_count = 0
    errored = 0

    for idx, fname in enumerate(src_files, start=1):
        src_path = os.path.join(src_folder, fname)
        print(f"[{idx}/{total}] Processing: {fname}")

        try:
            # Compute fingerprint for the source file
            src_fp = _xml_semantic_fingerprint(src_path,
                                               ignore_whitespace=ignore_whitespace,
                                               ignore_comments=ignore_comments,
                                               ignore_child_order=ignore_child_order)

            if src_fp in dest_fingerprints:
                print("  Skipping (identical content exists in destination)")
                skipped_count += 1
                continue

            # No identical content found → copy, avoid overwriting
            dest_path = os.path.join(dest_folder, fname)
            base, extname = os.path.splitext(fname)
            counter = 1
            while os.path.exists(dest_path):
                dest_path = os.path.join(dest_folder, f"{base}##{counter}{extname}")
                counter += 1

            shutil.copy2(src_path, dest_path)
            print(f"  Copied -> {dest_path}")
            copied_count += 1

            # Add fingerprint of the newly copied file
            dest_fingerprints.add(src_fp)

        except Exception as e:
            print(f"  Error handling {fname}: {e}")
            errored += 1

    print(f"\nSummary: {copied_count} copied, {skipped_count} skipped (identical), {errored} errors")


# Example

In [22]:
src_folder = '/Users/taitinglu/Desktop/IMG2SCH//Github Repo/Adafruit Sch Cleaned'
dest_folder = '/Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned'

copy_unique_xml_files_with_semantic_compare(
    src_folder,
    dest_folder
)

[1/895] Processing: Adafruit LIS3MDL + LSM6DS33.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/Adafruit LIS3MDL + LSM6DS33.sch
[2/895] Processing: Adafruit OLED FeatherWing.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/Adafruit OLED FeatherWing.sch
[3/895] Processing: adafruit flora mainboard 2.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/adafruit flora mainboard 2.sch
[4/895] Processing: Adafruit Feather ESP32-S2 Rev C.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/Adafruit Feather ESP32-S2 Rev C.sch
[5/895] Processing: Adafruit DotStar Wing.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/Adafruit DotStar Wing.sch
[6/895] Processing: Adafruit CLUE nRF52840 Express.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/Adafruit CLUE nRF52840 Express.sch
[7/895] Processing: DoubleWing.sch
  Copied -> /Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned/DoubleWing.sch
[8/895] Processing: Adafruit Metro ESP32-S3 Rev B.s

# version control: old eagle version to 9.6.2

In [13]:
import os
import csv
import glob
import re

def check_eagle_version(file_path, target_version="9.6.2"):
    """
    Return True if the .sch file contains eagle version="target_version", else False.
    Reads safely with errors='ignore' and stops early when a version tag is found.
    """
    version_pat = re.compile(r'version\s*=\s*"([^"]+)"', re.IGNORECASE)
    eagle_pat = re.compile(r'\beagle\b', re.IGNORECASE)

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # Heuristic: only check lines that mention 'eagle' to speed up
                if 'eagle' not in line.lower():
                    continue
                if eagle_pat.search(line):
                    m = version_pat.search(line)
                    if m:
                        return (m.group(1) == target_version)
        # If we never found a version tag, treat as outdated
        return False
    except Exception:
        # If unreadable, treat as outdated so it shows up in CSV for manual review
        return False


def export_outdated_eagle_schematics(folder_path,
                                     output_csv,
                                     target_version="9.6.2"):
    """
    Scan `folder_path` recursively for .sch files, find ones not matching the target Eagle version,
    and save them to `output_csv`. Returns the list of outdated file paths.

    Args:
        folder_path (str): Root folder to scan (recursively)
        output_csv (str): CSV file path to write results
        target_version (str): Expected Eagle version string (default "9.6.2")

    Returns:
        list[str]: Outdated .sch file paths
    """
    if not os.path.isdir(folder_path):
        raise ValueError(f"Folder not found: {folder_path}")

    # Collect all .sch files recursively
    sch_files = glob.glob(os.path.join(folder_path, "**", "*.sch"), recursive=True)

    outdated_files = []
    for sch in sch_files:
        ok = check_eagle_version(sch, target_version=target_version)
        if not ok:
            outdated_files.append(sch)

    # Ensure output directory exists
    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)

    # Write CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Outdated Schematic Files", f"Expected version={target_version}"])
        for path in outdated_files:
            writer.writerow([path])

    print(f"Scanned {len(sch_files)} .sch files.")
    print(f"Found {len(outdated_files)} outdated files.")
    print(f"Saved list to: {output_csv}")

    return outdated_files



# Example

In [14]:
out_csv = "/Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv"
export_outdated_eagle_schematics(
    folder_path="/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned",
    output_csv=out_csv,
    target_version="9.6.2"
)


Scanned 909 .sch files.
Found 551 outdated files.
Saved list to: /Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv


['/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit LIS3MDL + LSM6DS33.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit OLED FeatherWing.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/adafruit flora mainboard 2.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit DotStar Wing.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit CLUE nRF52840 Express.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/DoubleWing.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit Circuit Playground Bluefruit.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit Ultimate GPS USB Micro B.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/JST2.sch',
 '/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Metro M4 Express AirLift.sch',
 '/Users/tait

# save sch to 9.6.2 version

In [ ]:
#!/usr/bin/env python3
import csv
import os
import subprocess
from typing import List, Tuple

def run_eagle_batch_from_csv(
    csv_file: str,
    eagle_path: str,
    ulp_path: str,
    has_header: bool = True,
    column_index: int = 0,
    filter_ext: Tuple[str, ...] = (".sch",),
    dry_run: bool = False,
) -> Tuple[List[str], List[str]]:
    """
    Read file paths from a CSV and run an Eagle ULP on each file.

    Args:
        csv_file (str): Path to CSV file containing file paths (one per row/column_index).
        eagle_path (str): Path to Eagle binary (e.g., '/Applications/EAGLE-9.6.2/eagle.app/Contents/MacOS/Eagle').
        ulp_path (str): Path to the ULP to run.
        has_header (bool): If True, skip the first row of the CSV.
        column_index (int): Which CSV column contains the file path (default first column).
        filter_ext (Tuple[str,...]): Only process files with these extensions (case-insensitive). Use () to disable.
        dry_run (bool): If True, print commands but do not execute.

    Returns:
        (successes, failures): lists of processed file paths
    """
    if not os.path.isfile(csv_file):
        raise FileNotFoundError(f"CSV not found: {csv_file}")
    if not os.path.isfile(eagle_path):
        raise FileNotFoundError(f"Eagle binary not found: {eagle_path}")
    if not os.path.isfile(ulp_path):
        raise FileNotFoundError(f"ULP not found: {ulp_path}")

    # Load paths from CSV
    files: List[str] = []
    with open(csv_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        if has_header:
            next(reader, None)
        for row in reader:
            if not row or len(row) <= column_index:
                continue
            path = row[column_index].strip()
            if not path:
                continue
            if filter_ext:
                if not path.lower().endswith(tuple(ext.lower() for ext in filter_ext)):
                    continue
            files.append(path)

    print(f"Found {len(files)} file(s) in CSV to process")

    successes: List[str] = []
    failures: List[str] = []

    for i, sch_file in enumerate(files, 1):
        print(f"[{i}/{len(files)}] Processing: {sch_file}")

        cmd = [
            eagle_path,
            "-C",
            f"RUN {ulp_path}; QUIT;",
            sch_file,
        ]

        if dry_run:
            # print("  (dry-run) Command:", " ".join(cmd))
            successes.append(sch_file)
            continue

        try:
            result = subprocess.run(cmd, check=True)
            print(f"  ✓ Success")
            successes.append(sch_file)
        except subprocess.CalledProcessError as e:
            # print(f"  ✗ Failed (return code: {e.returncode})")
            failures.append(sch_file)
        except Exception as e:
            # print(f"  ✗ Error: {e}")
            failures.append(sch_file)

    print("\n=== Summary ===")
    print(f"Success: {len(successes)}")
    print(f"Failed : {len(failures)}")
    if failures:
        print("Failed files:")
        for p in failures:
            print("  -", p)

    return successes, failures



def remove_non_sch_files(folder_path: str):
    """
    Remove all files in `folder_path` whose extension is not `.sch`.
    Does not touch subfolders.

    Args:
        folder_path (str): Path to the folder to clean.
    """
    if not os.path.isdir(folder_path):
        raise ValueError(f"Not a valid folder: {folder_path}")

    removed_count = 0
    skipped_count = 0

    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        if os.path.isfile(fpath):
            if not fname.lower().endswith(".sch"):
                try:
                    os.remove(fpath)
                    removed_count += 1
                    print(f"Deleted: {fpath}")
                except Exception as e:
                    print(f"Failed to delete {fpath}: {e}")
            else:
                skipped_count += 1

    print(f"\nSummary: Removed {removed_count} file(s), kept {skipped_count} .sch file(s)")


# Example

In [16]:
success, fail = run_eagle_batch_from_csv(
    csv_file="/Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv",
    eagle_path="/Applications/EAGLE-9.6.2/eagle.app/Contents/MacOS/Eagle",
    ulp_path="/Users/taitinglu/Desktop/IMG2SCH/dummy_edit.ulp",
    has_header=False,          # set True if your CSV has a header
    column_index=0,            # first column contains paths
    filter_ext=(".sch",),      # only process .sch files
    dry_run=False              # set True to preview commands
)


Found 551 file(s) in CSV to process
[1/551] Processing: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit LIS3MDL + LSM6DS33.sch


2025-08-11 01:58:14.793 Eagle[60910:9171673] +[IMKClient subclass]: chose IMKClient_Modern
2025-08-11 01:58:14.793 Eagle[60910:9171673] +[IMKInputSession subclass]: chose IMKInputSession_Modern
QObject::disconnect: Unexpected null parameter
QObject::disconnect: Unexpected null parameter


  ✓ Success
[2/551] Processing: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit OLED FeatherWing.sch


2025-08-11 01:58:16.840 Eagle[60939:9171994] +[IMKClient subclass]: chose IMKClient_Modern
2025-08-11 01:58:16.840 Eagle[60939:9171994] +[IMKInputSession subclass]: chose IMKInputSession_Modern
QObject::disconnect: Unexpected null parameter
QObject::disconnect: Unexpected null parameter


  ✓ Success
[3/551] Processing: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/adafruit flora mainboard 2.sch


...

  ✓ Success
[550/551] Processing: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/adafruit flora neopixel.sch


2025-08-11 02:20:30.349 Eagle[67843:9246594] +[IMKClient subclass]: chose IMKClient_Modern
2025-08-11 02:20:30.349 Eagle[67843:9246594] +[IMKInputSession subclass]: chose IMKInputSession_Modern
Type conversion already registered from type QSharedPointer<QNetworkSession> to type QObject*
QObject::disconnect: Unexpected null parameter
QObject::disconnect: Unexpected null parameter
js: Uncaught SyntaxError: Unexpected token import
onmessage is not a callable property of qt.webChannelTransport. Some things might not work as expected.
js: Uncaught SyntaxError: Unexpected token import


  ✓ Success
[551/551] Processing: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit 2.13in Tri-Color eInk Display.sch


2025-08-11 02:20:34.128 Eagle[67852:9246741] +[IMKClient subclass]: chose IMKClient_Modern
2025-08-11 02:20:34.128 Eagle[67852:9246741] +[IMKInputSession subclass]: chose IMKInputSession_Modern
QObject::disconnect: Unexpected null parameter
QObject::disconnect: Unexpected null parameter


  ✓ Success

=== Summary ===
Success: 551
Failed : 0


In [21]:
out_csv = "/Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv"
export_outdated_eagle_schematics(
    folder_path="/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Protocentral Sch Cleaned",
    output_csv=out_csv,
    target_version="9.6.2"
)

remove_non_sch_files("/Users/taitinglu/Desktop/IMG2SCH/Github Repo/Protocentral Sch Cleaned")


Scanned 69 .sch files.
Found 0 outdated files.
Saved list to: /Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv

Summary: Removed 0 file(s), kept 69 .sch file(s)


# Remove Outdated

In [19]:
import csv
from pathlib import Path

def remove_outdated_schematics(csv_path: str,
                               strict_ext: bool = True,
                               dry_run: bool = False):
    """
    Remove outdated .sch files listed in a CSV.

    CSV requirements:
      - Must have a column named exactly "Outdated Schematic Files"
        (case-insensitive match allowed). The values are full file paths.

    Args:
        csv_path: path to the CSV file.
        strict_ext: if True, only delete files with .sch extension.
        dry_run: if True, only report what would be deleted.

    Returns:
        dict with counts: {'deleted': int, 'missing': int, 'skipped': int, 'rows': int}
    """
    csv_path = Path(csv_path)

    # Load rows
    with csv_path.open(newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        # find the target column
        col = None
        for name in (reader.fieldnames or []):
            if name and name.strip().lower().startswith("outdated schematic files"):
                col = name
                break
        if not col:
            raise ValueError("CSV must have a column named 'Outdated Schematic Files'.")

        rows = list(reader)

    deleted = missing = skipped = 0
    total = len(rows)

    for idx, row in enumerate(rows, start=1):
        file_path = (row.get(col) or "").strip()
        if not file_path:
            skipped += 1
            print(f"[{idx}/{total}] Empty path -> skipped")
            continue

        p = Path(file_path)

        if strict_ext and p.suffix.lower() != ".sch":
            skipped += 1
            print(f"[{idx}/{total}] Not .sch -> skipped: {p}")
            continue

        if not p.exists():
            missing += 1
            print(f"[{idx}/{total}] Missing -> {p}")
            continue

        if dry_run:
            print(f"[{idx}/{total}] (dry-run) Would delete: {p}")
            continue

        try:
            p.unlink()
            deleted += 1
            print(f"[{idx}/{total}] Deleted: {p}")
        except Exception as e:
            skipped += 1
            print(f"[{idx}/{total}] FAILED ({e}): {p}")

    summary = {"deleted": deleted, "missing": missing, "skipped": skipped, "rows": total}
    print(f"\nDone. Deleted {deleted}, missing {missing}, skipped {skipped} (rows: {total}).")
    return summary



In [20]:
out_csv = "/Users/taitinglu/Desktop/IMG2SCH/outdated_sch_files.csv"
remove_outdated_schematics(out_csv, dry_run=False)

[1/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/pmod_qspi_psram.sch
[2/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/fpga.sch
[3/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/feather_ice40.sch
[4/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/pmod_hyperram.sch
[5/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/pico-dev_1.sch
[6/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/Adafruit LM4040.sch
[7/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/feather_five.sch
[8/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/calculator2.sch
[9/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/picodvi_pmod.sch
[10/14] Deleted: /Users/taitinglu/Desktop/IMG2SCH/Github Repo/Adafruit Sch Cleaned/picodvi_1.sch
[11/14] Deleted

{'deleted': 14, 'missing': 0, 'skipped': 0, 'rows': 14}